In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def get_beam_waist(w_z, z, z0, wavelength):
    """Calculates w0 given w(z), z, and z0."""
    # Derived from w(z)^2 = w0^2 + (lambda^2 * (z-z0)^2) / (pi^2 * w0^2)
    # This is a quadratic in w0^2: (w0^2)^2 - w(z)^2 * w0^2 + [lambda*(z-z0)/pi]^2 = 0
    term = (wavelength * (z - z0) / np.pi)**2
    discriminant = w_z**4 - 4 * term
    if discriminant < 0:
        return None
    # Return the larger root (physical waist)
    return np.sqrt((w_z**2 + np.sqrt(discriminant)) / 2)

class GaussianApp:
    def __init__(self):
        self.output = widgets.Output()
        
        # UI Elements for X and Y
        self.inputs = {}
        dim = 'x'
        self.inputs[dim] = {
            'w0': self.create_row(f"Waist $w_0$ {dim} [µm]", 1.0),
            'z0': self.create_row(f"Focus Pos $z_0$ {dim} [mm]", 0.0),
            'wl': self.create_row(f"Wavelength [nm]", 1.0, fixed=True),
            'z_test': self.create_row(f"Test Pos $z$ [mm]", -2600.0),
            'w_test': self.create_row(f"Size at $z$ [µm]", 500.0)
        }
        dim = 'y'
        self.inputs[dim] = {
            'w0': self.create_row(f"Waist $w_0$ {dim} [µm]", 1.0),
            'z0': self.create_row(f"Focus Pos $z_0$ {dim} [mm]", 0.0),
            'wl': self.create_row(f"Wavelength [nm]", 1.0, fixed=True),
            'z_test': self.create_row(f"Test Pos $z$ [mm]", -3350.0),
            'w_test': self.create_row(f"Size at $z$ [µm]", 500.0)
        }

        # Layout
        self.ui = widgets.VBox([
            widgets.HBox([self.create_panel('X-Dimension', self.inputs['x']), 
                          self.create_panel('Y-Dimension', self.inputs['y'])]),
            widgets.Button(description="Update Plot", button_style='primary')
        ])
        self.ui.children[-1].on_click(self.update)
        
    def create_row(self, label, value, fixed=False):
        float_val = widgets.FloatText(value=value, layout={'width': '100px'})
        check = widgets.Checkbox(value=fixed, description='Fix', layout={'width': '80px'})
        return widgets.HBox([widgets.Label(label, layout={'width': '150px'}), float_val, check])

    def create_panel(self, title, group):
        return widgets.VBox([widgets.HTML(f"<b>{title}</b>")] + list(group.values()))

    def calculate_params(self, dim):
        data = self.inputs[dim]
        wl = data['wl'].children[1].value * 1e-6 # nm to mm
        z0 = data['z0'].children[1].value
        w0 = data['w0'].children[1].value * 1e-3 # um to mm
        
        z_t = data['z_test'].children[1].value
        w_t = data['w_test'].children[1].value * 1e-3
        
        # Solver Logic: If w_test is fixed, re-calculate w0 or z0
        if data['w_test'].children[2].value:
            # If we fix size at Z, and z0 is NOT fixed, we move focus to satisfy
            if not data['z0'].children[2].value:
                # Solve for z0: (z-z0)^2 = (w^2 - w0^2) * (pi*w0/lambda)^2
                zr = (np.pi * w0**2) / wl
                dz = np.sqrt(max(0, (w_t/w0)**2 - 1)) * zr
                z0 = z_t - dz # Move focus
                data['z0'].children[1].value = z0
            else:
                # If z0 is fixed but w_test is fixed, we solve for a new w0
                new_w0 = get_beam_waist(w_t, z_t, z0, wl)
                if new_w0:
                    w0 = new_w0
                    data['w0'].children[1].value = w0 * 1e3

        return w0, z0, wl

    def update(self, _=None):
        with self.output:
            clear_output(wait=True)
            plt.figure(figsize=(10, 5))
            z_range = np.linspace(-3500, 4000, 1000)
            
            for dim, color in zip(['x', 'y'], ['blue', 'red']):
                w0, z0, wl = self.calculate_params(dim)
                zr = (np.pi * w0**2) / wl
                w_z = w0 * np.sqrt(1 + ((z_range - z0) / zr)**2)
                
                plt.plot(z_range, w_z * 1e3, color=color, label=f'Beam {dim}')
                plt.plot(z_range, -w_z * 1e3, color=color, alpha=0.3)
                plt.fill_between(z_range, -w_z * 1e3, w_z * 1e3, color=color, alpha=0.1)

            plt.xlabel("z [mm]")
            plt.ylabel("Beam Radius [µm]")
            plt.grid(True, linestyle=':')
            plt.legend()
            plt.show()

app = GaussianApp()
display(app.ui, app.output)

Output()

In [2]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def get_beam_waist(w_z, z, z0, wavelength):
    term = (wavelength * (z - z0) / np.pi)**2
    discriminant = w_z**4 - 4 * term
    if discriminant < 0: return None
    return np.sqrt((w_z**2 + np.sqrt(discriminant)) / 2)

class GaussianApp:
    def __init__(self):
        # Configuration for FWHM constant
        self.fwhm_factor = np.sqrt(2 * np.log(2))
        
        # UI Elements
        self.inputs = {}
        for dim in ['x', 'y']:
            z_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'w0': self.create_row(f"Waist $w_0$ {dim} [µm]", 1.0),
                'z0': self.create_row(f"Focus Pos $z_0$ {dim} [mm]", 0.0),
                'wl': self.create_row(f"Wavelength [nm]", 1.0, fixed=True),
                'z_test': self.create_row(f"Test Pos $z$ [mm]", z_def),
                'w_test': self.create_row(f"Size at $z$ [µm]", 500.0)
            }

        self.log_check = widgets.Checkbox(value=False, description='Log Scale (Y-axis)')
        self.btn = widgets.Button(description="Update Plot", button_style='primary')
        self.btn.on_click(self.update)
        
        # Initialize Plot
        plt.ioff() # Turn off interactive mode to handle display manually
        self.fig, self.ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
        self.canvas = self.fig.canvas
        
        self.ui = widgets.VBox([
            widgets.HBox([self.create_panel('X-Dimension', self.inputs['x']), 
                          self.create_panel('Y-Dimension', self.inputs['y'])]),
            widgets.HBox([self.log_check, self.btn]),
            self.canvas
        ])
        
    def create_row(self, label, value, fixed=False):
        float_val = widgets.FloatText(value=value, layout={'width': '100px'})
        check = widgets.Checkbox(value=fixed, description='Fix', layout={'width': '70px'})
        return widgets.HBox([widgets.Label(label, layout={'width': '150px'}), float_val, check])

    def create_panel(self, title, group):
        return widgets.VBox([widgets.HTML(f"<b>{title}</b>")] + list(group.values()))

    def calculate_params(self, dim):
        data = self.inputs[dim]
        wl = data['wl'].children[1].value * 1e-6
        z0 = data['z0'].children[1].value
        w0 = data['w0'].children[1].value * 1e-3
        z_t, w_t = data['z_test'].children[1].value, data['w_test'].children[1].value * 1e-3
        
        if data['w_test'].children[2].value:
            if not data['z0'].children[2].value:
                zr = (np.pi * w0**2) / wl
                if w_t > w0:
                    z0 = z_t - (np.sqrt((w_t/w0)**2 - 1) * zr)
                    data['z0'].children[1].value = round(z0, 2)
            else:
                new_w0 = get_beam_waist(w_t, z_t, z0, wl)
                if new_w0:
                    w0 = new_w0
                    data['w0'].children[1].value = round(w0 * 1e3, 4)
        return w0, z0, wl

    def update(self, _=None):
        self.ax.clear()
        z_range = np.linspace(-4000, 4000, 2000)
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            w0, z0, wl = self.calculate_params(dim)
            zr = (np.pi * w0**2) / wl
            w_z = w0 * np.sqrt(1 + ((z_range - z0) / zr)**2)
            fwhm_z = w_z * self.fwhm_factor
            
            # Plot 1/e^2 radius
            self.ax.plot(z_range, w_z * 1e3, color=color, linewidth=2, 
                         label=f'{dim}: $1/e^2$ radius')
            # Plot FWHM
            self.ax.plot(z_range, fwhm_z * 1e3, color=color, linestyle='--', alpha=0.7,
                         label=f'{dim}: FWHM')
            
            # Constraints
            if self.inputs[dim]['w_test'].children[2].value:
                self.ax.scatter([self.inputs[dim]['z_test'].children[1].value], 
                                [self.inputs[dim]['w_test'].children[1].value], 
                                color=color, marker='o', s=40, zorder=5)

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_xlabel("Propagating Axis $z$ [mm]")
        self.ax.set_ylabel("Beam Extent [µm]")
        self.ax.set_title("Gaussian Beam Evolution (Single-Sided)")
        self.ax.legend(fontsize='small', ncol=2)
        self.ax.grid(True, which="both", ls="-", alpha=0.2)
        self.canvas.draw()

app = GaussianApp()
display(app.ui)

In [1]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def get_beam_waist(w_z, z, z0, wavelength):
    term = (wavelength * (z - z0) / np.pi)**2
    discriminant = w_z**4 - 4 * term
    if discriminant < 0: return None
    return np.sqrt((w_z**2 + np.sqrt(discriminant)) / 2)

class GaussianApp:
    def __init__(self):
        self.fwhm_factor = np.sqrt(2 * np.log(2))
        self.points_per_mm = 10
        
        # Core Parameters
        self.inputs = {}
        for dim in ['x', 'y']:
            z_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'w0': self.create_input(1.0),
                'z0': self.create_input(0.0),
                'wl': self.create_input(1.0, fixed=True),
                'z_test': self.create_input(z_def),
                'w_test': self.create_input(500.0)
            }

        # Dynamic Rows for Z-Positions
        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        self.add_row_btn = widgets.Button(description="+ Add Z-Point", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove", button_style='danger')
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        
        self.log_check = widgets.Checkbox(value=False, description='Log Scale')
        self.log_check.observe(self.update, 'value')

        # Layout Building
        header = widgets.HTML("<b>Gaussian Beam Solver</b>")
        params_ui = widgets.HBox([self.create_panel('X-Dim', self.inputs['x']), 
                                  self.create_panel('Y-Dim', self.inputs['y'])])
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            header, params_ui, 
            widgets.HTML("<b>Dynamic Z-Probes</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.log_check]),
            self.canvas
        ])
        
        self.add_z_row(0.0) # Start with one row
        self.update()

    def create_input(self, value, fixed=False):
        w = widgets.FloatText(value=value, layout={'width': '90px'})
        c = widgets.Checkbox(value=fixed, layout={'width': '30px'})
        w.observe(self.update, 'value')
        c.observe(self.update, 'value')
        return {'val': w, 'fix': c}

    def create_panel(self, title, group):
        rows = [widgets.HTML(f"<i>{title}</i>")]
        labels = ["Waist $w_0$ [µm]", "Focus $z_0$ [mm]", "Wave [nm]", "Test $z$ [mm]", "Size at $z$ [µm]"]
        for label, (key, item) in zip(labels, group.items()):
            rows.append(widgets.HBox([widgets.Label(label, layout={'width': '120px'}), item['val'], item['fix']]))
        return widgets.VBox(rows, layout={'border': '1px solid #ddd', 'padding': '5px'})

    def add_z_row(self, initial_z=100.0):
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '100px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '100px'})
        z_in.observe(self.update, 'value')
        row = widgets.HBox([widgets.Label("Pos Z:"), z_in, widgets.Label("X size:"), x_out, widgets.Label("Y size:"), y_out])
        self.z_rows.append({'row': row, 'z': z_in, 'x': x_out, 'y': y_out})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update()

    def remove_z_row(self, _=None):
        if len(self.z_rows) > 0:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update()

    def calculate_beam(self, dim):
        d = self.inputs[dim]
        wl = d['wl']['val'].value * 1e-6
        z0, w0 = d['z0']['val'].value, d['w0']['val'].value * 1e-3
        zt, wt = d['z_test']['val'].value, d['w_test']['val'].value * 1e-3
        
        if d['w_test']['fix'].value:
            if not d['z0']['fix'].value:
                zr = (np.pi * w0**2) / wl
                if wt > w0:
                    z0 = zt - (np.sqrt((wt/w0)**2 - 1) * zr)
                    d['z0']['val'].value = round(z0, 2)
            else:
                new_w0 = get_beam_waist(wt, zt, z0, wl)
                if new_w0:
                    w0 = new_w0
                    d['w0']['val'].value = round(w0 * 1e3, 4)
        return w0, z0, wl

    def update(self, _=None):
        self.ax.clear()
        z_min, z_max = -4000, 4000
        z_range = np.linspace(z_min, z_max, int((z_max - z_min) * self.points_per_mm))
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            w0, z0, wl = self.calculate_beam(dim)
            zr = (np.pi * w0**2) / wl
            w_z_func = lambda z: w0 * np.sqrt(1 + ((z - z0) / zr)**2)
            
            w_vals = w_z_func(z_range)
            self.ax.plot(z_range, w_vals * 1e3, color=color, label=f'{dim} $1/e^2$')
            self.ax.plot(z_range, w_vals * self.fwhm_factor * 1e3, color=color, ls='--', alpha=0.5, label=f'{dim} FWHM')
            
            # Update the Dynamic Z-Probe outputs
            for row in self.z_rows:
                size = w_z_func(row['z'].value) * 1e3
                if dim == 'x': row['x'].value = round(size, 2)
                else: row['y'].value = round(size, 2)
                self.ax.axvline(row['z'].value, color='gray', lw=0.5, ls=':')

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Radius [µm]")
        self.ax.legend(fontsize='x-small', ncol=2)
        self.ax.grid(True, which="both", alpha=0.3)
        self.canvas.draw()

app = GaussianApp()
display(app.ui)

In [1]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Javascript
import pandas as pd
import io

def get_beam_waist(w_z, z, z0, wavelength):
    term = (wavelength * (z - z0) / np.pi)**2
    discriminant = w_z**4 - 4 * term
    if discriminant < 0: return None
    return np.sqrt((w_z**2 + np.sqrt(discriminant)) / 2)

class GaussianApp:
    def __init__(self):
        self.fwhm_factor = np.sqrt(2 * np.log(2))
        self.points_per_mm = 10
        
        # Core Parameters
        self.inputs = {}
        for dim in ['x', 'y']:
            z_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'w0': self.create_input(1.0),
                'z0': self.create_input(0.0),
                'wl': self.create_input(1.0, fixed=True),
                'z_test': self.create_input(z_def),
                'w_test': self.create_input(500.0)
            }

        # Dynamic Rows for Z-Positions
        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        
        self.add_row_btn = widgets.Button(description="+ Add Z-Point", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        self.log_check = widgets.Checkbox(value=False, description='Log Scale', layout={'width': '150px'})
        self.log_check.observe(self.update, 'value')

        # Layout Building
        params_ui = widgets.HBox([self.create_panel('X-Dimension (Horizontal)', self.inputs['x']), 
                                  self.create_panel('Y-Dimension (Vertical)', self.inputs['y'])])
        
        plt.ioff()
        # Adjusted figure size to accommodate legend on the right
        self.fig, self.ax = plt.subplots(figsize=(11, 5))
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            widgets.HTML("<h2>Gaussian Beam Solver & Beamline Component Tracker</h2>"),
            params_ui, 
            widgets.HTML("<b>Beamline Components (Z-Probes)</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn, self.log_check]),
            self.canvas
        ])
        
        # Add default rows
        defaults = [
            ("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
            ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
            ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
            ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)
        ]
        for name, z_pos in defaults:
            self.add_z_row(z_pos, name)
            
        self.update()

    def create_input(self, value, fixed=False):
        w = widgets.FloatText(value=value, layout={'width': '90px'})
        c = widgets.Checkbox(value=fixed, layout={'width': '30px'})
        w.observe(self.update, 'value')
        c.observe(self.update, 'value')
        return {'val': w, 'fix': c}

    def create_panel(self, title, group):
        rows = [widgets.HTML(f"<b>{title}</b>")]
        labels = ["Waist $w_0$ [µm]", "Focus $z_0$ [mm]", "Wave [nm]", "Test $z$ [mm]", "Size at $z$ [µm]"]
        for label, (key, item) in zip(labels, group.items()):
            rows.append(widgets.HBox([widgets.Label(label, layout={'width': '120px'}), item['val'], item['fix']]))
        return widgets.VBox(rows, layout={'border': '1px solid #ddd', 'padding': '10px', 'margin': '5px'})

    def add_z_row(self, initial_z=0.0, initial_name="New Point"):
        name_in = widgets.Text(value=initial_name, placeholder='Component Name', layout={'width': '150px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        
        for w in [name_in, z_in]: w.observe(self.update, 'value')
        
        row_ui = widgets.HBox([widgets.Label("Name:"), name_in, widgets.Label("Z:"), z_in, 
                               widgets.Label("X size:"), x_out, widgets.Label("Y size:"), y_out])
        
        self.z_rows.append({'row': row_ui, 'name': name_in, 'z': z_in, 'x': x_out, 'y': y_out})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update()

    def remove_z_row(self, _=None):
        if len(self.z_rows) > 0:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update()

    def export_to_csv(self, _=None):
        data = []
        for r in self.z_rows:
            data.append([r['name'].value, r['z'].value, r['x'].value, r['y'].value])
        df = pd.DataFrame(data, columns=['Component', 'Z_Pos_mm', 'X_Radius_um', 'Y_Radius_um'])
        
        # Simple print-to-output or standard download trick for Jupyter
        filename = "beam_profile_export.csv"
        df.to_csv(filename, index=False)
        print(f"Success: Exported {len(data)} rows to {filename}")

    def calculate_beam(self, dim):
        d = self.inputs[dim]
        wl = d['wl']['val'].value * 1e-6
        z0, w0 = d['z0']['val'].value, d['w0']['val'].value * 1e-3
        zt, wt = d['z_test']['val'].value, d['w_test']['val'].value * 1e-3
        
        if d['w_test']['fix'].value:
            if not d['z0']['fix'].value:
                zr = (np.pi * w0**2) / wl
                if wt > w0:
                    z0 = zt - (np.sqrt((wt/w0)**2 - 1) * zr)
                    d['z0']['val'].value = round(z0, 2)
            else:
                new_w0 = get_beam_waist(wt, zt, z0, wl)
                if new_w0:
                    w0 = new_w0
                    d['w0']['val'].value = round(w0 * 1e3, 4)
        return w0, z0, wl

    def update(self, _=None):
        self.ax.clear()
        z_min, z_max = -4500, 4500
        z_range = np.linspace(z_min, z_max, int((z_max - z_min) * self.points_per_mm))
        
        # Plot Beam Envelopes
        for dim, color, lbl in zip(['x', 'y'], ['#1f77b4', '#d62728'], ['Hor (X)', 'Ver (Y)']):
            w0, z0, wl = self.calculate_beam(dim)
            zr = (np.pi * w0**2) / wl
            w_z_func = lambda z: w0 * np.sqrt(1 + ((z - z0) / zr)**2)
            
            w_vals = w_z_func(z_range)
            self.ax.plot(z_range, w_vals * 1e3, color=color, lw=2, label=f'{lbl} $1/e^2$')
            self.ax.plot(z_range, w_vals * self.fwhm_factor * 1e3, color=color, ls='--', alpha=0.4, label=f'{lbl} FWHM')
            
            # Update values in the UI rows
            for row in self.z_rows:
                size = w_z_func(row['z'].value) * 1e3
                if dim == 'x': row['x'].value = round(size, 2)
                else: row['y'].value = round(size, 2)

        # Draw Component Vertical Lines
        # Use a colormap to distinguish multiple vertical lines
        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            self.ax.axvline(row['z'].value, color=colors(i), lw=1.5, ls='-', alpha=0.8, label=row['name'].value)

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_xlabel("Propagating Axis $z$ [mm]")
        self.ax.set_ylabel("Beam Radius [µm]")
        self.ax.grid(True, which="both", alpha=0.2)
        
        # Place legend outside to the right
        self.ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', borderaxespad=0.)
        self.canvas.draw()

app = GaussianApp()
display(app.ui)

/tmp/ipykernel_491551/1956557349.py:165: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))


In [1]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

def energy_to_wavelength_mm(energy_kev):
    # lambda [m] = h*c / E
    # lambda [mm] = 1.23984193e-6 / energy_kev
    return 1.23984193e-6 / energy_kev

def get_beam_waist_from_fwhm(fwhm_z, z, z0, wl_mm):
    """Calculates w0 (1/e2 radius) given FWHM(z), z, z0, and wavelength."""
    # Convert FWHM to 1/e2 radius: w(z) = FWHM / sqrt(2*ln2)
    w_z = fwhm_z / np.sqrt(2 * np.log(2))
    term = (wl_mm * (z - z0) / np.pi)**2
    discriminant = w_z**4 - 4 * term
    if discriminant < 0: return None
    return np.sqrt((w_z**2 + np.sqrt(discriminant)) / 2)

class GaussianApp:
    def __init__(self):
        self.fwhm_to_w = 1 / np.sqrt(2 * np.log(2))
        self.w_to_fwhm = np.sqrt(2 * np.log(2))
        self.points_per_mm = 10
        
        # Core Parameters
        self.inputs = {}
        for dim in ['x', 'y']:
            z_def = -3350.0 if dim == 'y' else -2600.0
            self.inputs[dim] = {
                'fwhm0': self.create_input(1.0),
                'z0': self.create_input(0.0),
                'energy': self.create_input(12.4, fixed=True), # Default ~1 Angstrom
                'z_test': self.create_input(z_def),
                'fwhm_test': self.create_input(500.0)
            }

        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        
        self.add_row_btn = widgets.Button(description="+ Add Z-Point", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        self.log_check = widgets.Checkbox(value=False, description='Log Scale', layout={'width': '150px'})
        self.log_check.observe(self.update, 'value')

        params_ui = widgets.HBox([self.create_panel('Horizontal (X)', self.inputs['x']), 
                                  self.create_panel('Vertical (Y)', self.inputs['y'])])
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(11, 6))
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            widgets.HTML("<h2>X-Ray Gaussian Beam Solver (FWHM / keV)</h2>"),
            params_ui, 
            widgets.HTML("<b>Beamline Components (Z-Probes)</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn, self.log_check]),
            self.canvas
        ])
        
        defaults = [
            ("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
            ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
            ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
            ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)
        ]
        for name, z_pos in defaults:
            self.add_z_row(z_pos, name)
            
        self.update()

    def create_input(self, value, fixed=False):
        w = widgets.FloatText(value=value, layout={'width': '90px'})
        c = widgets.Checkbox(value=fixed, layout={'width': '30px'})
        w.observe(self.update, 'value')
        c.observe(self.update, 'value')
        return {'val': w, 'fix': c}

    def create_panel(self, title, group):
        rows = [widgets.HTML(f"<b>{title}</b>")]
        labels = ["Waist FWHM [µm]", "Focus $z_0$ [mm]", "Energy [keV]", "Test $z$ [mm]", "FWHM at $z$ [µm]"]
        for label, (key, item) in zip(labels, group.items()):
            rows.append(widgets.HBox([widgets.Label(label, layout={'width': '130px'}), item['val'], item['fix']]))
        return widgets.VBox(rows, layout={'border': '1px solid #ddd', 'padding': '10px', 'margin': '5px'})

    def add_z_row(self, initial_z=0.0, initial_name="New Point"):
        name_in = widgets.Text(value=initial_name, layout={'width': '150px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        
        for w in [name_in, z_in]: w.observe(self.update, 'value')
        
        row_ui = widgets.HBox([widgets.Label("Name:"), name_in, widgets.Label("Z:"), z_in, 
                               widgets.Label("X FWHM:"), x_out, widgets.Label("Y FWHM:"), y_out])
        
        self.z_rows.append({'row': row_ui, 'name': name_in, 'z': z_in, 'x': x_out, 'y': y_out})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update()

    def remove_z_row(self, _=None):
        if len(self.z_rows) > 0:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update()

    def export_to_csv(self, _=None):
        data = [[r['name'].value, r['z'].value, r['x'].value, r['y'].value] for r in self.z_rows]
        df = pd.DataFrame(data, columns=['Component', 'Z_Pos_mm', 'X_FWHM_um', 'Y_FWHM_um'])
        df.to_csv("beam_fwhm_export.csv", index=False)
        print("Exported to beam_fwhm_export.csv")

    def calculate_beam(self, dim):
        d = self.inputs[dim]
        wl_mm = energy_to_wavelength_mm(d['energy']['val'].value)
        z0 = d['z0']['val'].value
        # Internal w0 is always 1/e2 radius in mm
        w0 = (d['fwhm0']['val'].value * 1e-3) * self.fwhm_to_w
        zt = d['z_test']['val'].value
        fwhm_t = d['fwhm_test']['val'].value
        
        if d['fwhm_test']['fix'].value:
            if not d['z0']['fix'].value:
                zr = (np.pi * w0**2) / wl_mm
                w_t_target = (fwhm_t * 1e-3) * self.fwhm_to_w
                if w_t_target > w0:
                    z0 = zt - (np.sqrt((w_t_target/w0)**2 - 1) * zr)
                    d['z0']['val'].value = round(z0, 2)
            else:
                new_w0 = get_beam_waist_from_fwhm(fwhm_t * 1e-3, zt, z0, wl_mm)
                if new_w0:
                    w0 = new_w0
                    d['fwhm0']['val'].value = round((w0 * self.w_to_fwhm) * 1e3, 4)
        return w0, z0, wl_mm

    def update(self, _=None):
        self.ax.clear()
        z_min, z_max = -4500, 4500
        z_range = np.linspace(z_min, z_max, int((z_max - z_min) * self.points_per_mm))
        
        for dim, color, lbl in zip(['x', 'y'], ['#1f77b4', '#d62728'], ['Hor (X)', 'Ver (Y)']):
            w0, z0, wl = self.calculate_beam(dim)
            zr = (np.pi * w0**2) / wl
            # Propagation func returns 1/e2 radius
            w_z_func = lambda z: w0 * np.sqrt(1 + ((z - z0) / zr)**2)
            
            # Plotting FWHM (solid) and 1/e2 (dashed/alpha)
            fwhm_vals = w_z_func(z_range) * self.w_to_fwhm * 1e3
            self.ax.plot(z_range, fwhm_vals, color=color, lw=2, label=f'{lbl} FWHM')
            self.ax.plot(z_range, w_z_func(z_range) * 1e3, color=color, ls='--', alpha=0.3, label=f'{lbl} $1/e^2$')
            
            for row in self.z_rows:
                fwhm_size = w_z_func(row['z'].value) * self.w_to_fwhm * 1e3
                if dim == 'x': row['x'].value = round(fwhm_size, 2)
                else: row['y'].value = round(fwhm_size, 2)

        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            self.ax.axvline(row['z'].value, color=colors(i), lw=1.5, alpha=0.7, label=row['name'].value)

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Beam Size [µm]")
        self.ax.grid(True, which="both", alpha=0.2)
        self.ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
        self.canvas.draw()

app = GaussianApp()
display(app.ui)

/tmp/ipykernel_489413/790298655.py:166: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))


In [9]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from mpl_interactions import ipyplot as iplt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

def energy_to_wavelength_mm(energy_kev):
    # lambda [m] = h*c / E
    # lambda [mm] = 1.23984193e-6 / energy_kev
    return 1.23984193e-6 / energy_kev

def get_beam_waist_from_fwhm(fwhm_z, z, z0, wl_mm):
    """Calculates w0 (1/e2 radius) given FWHM(z), z, z0, and wavelength."""
    # Convert FWHM to 1/e2 radius: w(z) = FWHM / sqrt(2*ln2)
    w_z = fwhm_z / np.sqrt(2 * np.log(2))
    term = (wl_mm * (z - z0) / np.pi)**2
    discriminant = w_z**4 - 4 * term
    if discriminant < 0: return None
    return np.sqrt((w_z**2 + np.sqrt(discriminant)) / 2)

class GaussianApp:
    def __init__(self):
        self.fwhm_to_w = 1 / np.sqrt(2 * np.log(2))
        self.w_to_fwhm = np.sqrt(2 * np.log(2))
        self.points_per_mm = 10
        
        # Core Parameters
        self.inputs = {}
        for dim in ['x', 'y']:
            z_def = -3350.0 if dim == 'y' else -2600.0
            self.inputs[dim] = {
                'fwhm0': self.create_input(1.0),
                'z0': self.create_input(0.0),
                'energy': self.create_input(12.4, fixed=True), # Default ~1 Angstrom
                'z_test': self.create_input(z_def),
                'fwhm_test': self.create_input(500.0)
            }

        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        
        self.add_row_btn = widgets.Button(description="+ Add Z-Point", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        self.log_check = widgets.Checkbox(value=False, description='Log Scale', layout={'width': '150px'})
        self.log_check.observe(self.update, 'value')

        params_ui = widgets.HBox([self.create_panel('Horizontal (X)', self.inputs['x']), 
                                  self.create_panel('Vertical (Y)', self.inputs['y'])])
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(11, 6),constrained_layout=True)
        self.lines_fwhm = {}
        self.lines_radius = {}
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            widgets.HTML("<h2>X-Ray Gaussian Beam Solver (FWHM / keV)</h2>"),
            params_ui, 
            widgets.HTML("<b>Beamline Components (Z-Probes)</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn, self.log_check]),
            self.canvas
        ])
        
        defaults = [
            ("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
            ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
            ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
            ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)
        ]
        for name, z_pos in defaults:
            self.add_z_row(z_pos, name)
            
        self.update()

    def create_input(self, value, fixed=False):b
        w = widgets.FloatText(value=value, layout={'width': '90px'})
        w.observe(self.update, 'value')
        return {'val': w}

    def create_panel(self, title, group):
        rows = [widgets.HTML(f"<b>{title}</b>")]
        labels = ["Waist FWHM [µm]", "Focus $z_0$ [mm]", "Energy [keV]", "Test $z$ [mm]", "FWHM at $z$ [µm]"]
        for label, (key, item) in zip(labels, group.items()):
            rows.append(widgets.HBox([widgets.Label(label, layout={'width': '130px'}), item['val']]))
        return widgets.VBox(rows, layout={'border': '1px solid #ddd', 'padding': '10px', 'margin': '5px'})

    def add_z_row(self, initial_z=0.0, initial_name="New Point"):
        name_in = widgets.Text(value=initial_name, layout={'width': '150px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        
        for w in [name_in, z_in]: w.observe(self.update, 'value')
        
        row_ui = widgets.HBox([widgets.Label("Name:"), name_in, widgets.Label("Z:"), z_in, 
                               widgets.Label("X FWHM:"), x_out, widgets.Label("Y FWHM:"), y_out])
        
        self.z_rows.append({'row': row_ui, 'name': name_in, 'z': z_in, 'x': x_out, 'y': y_out})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update()

    def remove_z_row(self, _=None):
        if len(self.z_rows) > 0:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update()

    def export_to_csv(self, _=None):
        data = [[r['name'].value, r['z'].value, r['x'].value, r['y'].value] for r in self.z_rows]
        df = pd.DataFrame(data, columns=['Component', 'Z_Pos_mm', 'X_FWHM_um', 'Y_FWHM_um'])
        df.to_csv("beam_fwhm_export.csv", index=False)
        print("Exported to beam_fwhm_export.csv")

    def calculate_beam(self, dim):
        d = self.inputs[dim]
        wl_mm = energy_to_wavelength_mm(d['energy']['val'].value)
        z0 = d['z0']['val'].value
        # Internal w0 is always 1/e2 radius in mm
        w0 = (d['fwhm0']['val'].value * 1e-3) * self.fwhm_to_w
        zt = d['z_test']['val'].value
        fwhm_t = d['fwhm_test']['val'].value
        
        
        
        zr = (np.pi * w0**2) / wl_mm
        w_t_target = (fwhm_t * 1e-3) * self.fwhm_to_w
        if w_t_target > w0:
            z0 = zt - (np.sqrt((w_t_target/w0)**2 - 1) * zr)
            d['z0']['val'].value = round(z0, 2)
            
        #     # else:
        # new_w0 = get_beam_waist_from_fwhm(fwhm_t * 1e-3, zt, z0, wl_mm)
        # if new_w0:
        #     w0 = new_w0
        #     d['fwhm0']['val'].value = round((w0 * self.w_to_fwhm) * 1e3, 4)
        return w0, z0, wl_mm

    def update(self, _=None):
        self.ax.clear()
        z_min, z_max = -4500, 4500
        z_range = np.linspace(z_min, z_max, int((z_max - z_min) * self.points_per_mm))
        
        for dim, color, lbl in zip(['x', 'y'], ['#1f77b4', '#d62728'], ['Hor (X)', 'Ver (Y)']):
            w0, z0, wl = self.calculate_beam(dim)
            
            zr = (np.pi * w0**2) / wl
            # Propagation func returns 1/e2 radius
            w_z_func = lambda z: w0 * np.sqrt(1 + ((z - z0) / zr)**2)
            
            # Plotting FWHM (solid) and 1/e2 (dashed/alpha)
            fwhm_vals = w_z_func(z_range) * self.w_to_fwhm * 1e3
            self.ax.plot(z_range, fwhm_vals, color=color, lw=2, label=f'{lbl} FWHM')[0]    
            self.ax.plot(z_range, w_z_func(z_range) * 1e3, color=color, ls='--', alpha=0.3, label=f'{lbl} $1/e^2$')[0]
            
                
            
            for row in self.z_rows:
                fwhm_size = w_z_func(row['z'].value) * self.w_to_fwhm * 1e3
                if dim == 'x': row['x'].value = round(fwhm_size, 2)
                else: row['y'].value = round(fwhm_size, 2)

        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            self.ax.axvline(row['z'].value, color=colors(i), lw=1.5, alpha=0.7, label=row['name'].value)

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Beam Size [µm]")
        self.ax.grid(True, which="both", alpha=0.2)
        self.ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
        # self.canvas.draw()
        self.canvas.draw_idle()

app = GaussianApp()
display(app.ui)

/tmp/ipykernel_511825/2133581450.py:170: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))


In [8]:
tmp=app.lines_fwhm['x']

In [15]:
plt.ylim(0,200)
plt.draw()

1.0332016083333334e-07

In [10]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

def energy_to_wavelength_mm(energy_kev):
    return 1.23984193e-6 / energy_kev

def get_beam_waist_from_fwhm(fwhm_z, z, z0, wl_mm):
    """Calculates w0 (1/e2 radius) given FWHM(z), z, z0, and wavelength."""
    w_z = fwhm_z / np.sqrt(2 * np.log(2))
    term = (wl_mm * (z - z0) / np.pi)**2
    discriminant = w_z**4 - 4 * term
    if discriminant < 0: return None
    return np.sqrt((w_z**2 + np.sqrt(discriminant)) / 2)

class GaussianApp:
    def __init__(self):
        self.fwhm_to_w = 1 / np.sqrt(2 * np.log(2))
        self.w_to_fwhm = np.sqrt(2 * np.log(2))
        self.points_per_mm = 10
        
        # Core Parameters
        self.inputs = {}
        for dim in ['x', 'y']:
            z_def = -3350.0 if dim == 'y' else -2600.0
            self.inputs[dim] = {
                'fwhm0': self.create_input(100.0),
                'z0': self.create_input(0.0),
                'energy': self.create_input(12.4, fixed=True),
                'z_test': self.create_input(z_def),
                'fwhm_test': self.create_input(500.0)
            }

        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        
        self.add_row_btn = widgets.Button(description="+ Add Z-Point", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        self.log_check = widgets.Checkbox(value=False, description='Log Scale', layout={'width': '150px'})
        self.log_check.observe(self.update, 'value')

        params_ui = widgets.HBox([self.create_panel('Horizontal (X)', self.inputs['x']), 
                                  self.create_panel('Vertical (Y)', self.inputs['y'])])
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            widgets.HTML("<h2>X-Ray Gaussian Beam Solver (FWHM / keV)</h2>"),
            params_ui, 
            widgets.HTML("<b>Beamline Components (Z-Probes)</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn, self.log_check]),
            self.canvas
        ])
        
        defaults = [
            ("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
            ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
            ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
            ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)
        ]
        for name, z_pos in defaults:
            self.add_z_row(z_pos, name)
            
        self.update()

    def create_input(self, value, fixed=False):
        w = widgets.FloatText(value=value, layout={'width': '90px'})
        c = widgets.Checkbox(value=fixed, layout={'width': '30px'})
        w.observe(self.update, 'value')
        c.observe(self.update, 'value')
        return {'val': w, 'fix': c}

    def create_panel(self, title, group):
        rows = [widgets.HTML(f"<b>{title}</b>")]
        labels = ["Waist FWHM [µm]", "Focus $z_0$ [mm]", "Energy [keV]", "Test $z$ [mm]", "FWHM at $z$ [µm]"]
        for label, (key, item) in zip(labels, group.items()):
            rows.append(widgets.HBox([widgets.Label(label, layout={'width': '130px'}), item['val'], item['fix']]))
        return widgets.VBox(rows, layout={'border': '1px solid #ddd', 'padding': '10px', 'margin': '5px'})

    def add_z_row(self, initial_z=0.0, initial_name="New Point"):
        name_in = widgets.Text(value=initial_name, layout={'width': '150px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        for w in [name_in, z_in]: w.observe(self.update, 'value')
        row_ui = widgets.HBox([widgets.Label("Name:"), name_in, widgets.Label("Z:"), z_in, 
                               widgets.Label("X FWHM:"), x_out, widgets.Label("Y FWHM:"), y_out])
        self.z_rows.append({'row': row_ui, 'name': name_in, 'z': z_in, 'x': x_out, 'y': y_out})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update()

    def remove_z_row(self, _=None):
        if len(self.z_rows) > 0:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update()

    def export_to_csv(self, _=None):
        data = [[r['name'].value, r['z'].value, r['x'].value, r['y'].value] for r in self.z_rows]
        df = pd.DataFrame(data, columns=['Component', 'Z_Pos_mm', 'X_FWHM_um', 'Y_FWHM_um'])
        df.to_csv("beam_fwhm_export.csv", index=False)

    def calculate_beam(self, dim):
        d = self.inputs[dim]
        wl_mm = energy_to_wavelength_mm(d['energy']['val'].value)
        z0 = d['z0']['val'].value
        w0 = (d['fwhm0']['val'].value * 1e-3) * self.fwhm_to_w
        zt = d['z_test']['val'].value
        fwhm_t = d['fwhm_test']['val'].value
        
        # Solver Logic: if fwhm_test is 'fixed' (checked), we solve for parameters
        if d['fwhm_test']['fix'].value:
            w_t_target = (fwhm_t * 1e-3) * self.fwhm_to_w
            # Case 1: Adjust Focus (z0) if z0 is not fixed
            if not d['z0']['fix'].value:
                zr = (np.pi * w0**2) / wl_mm
                if w_t_target > w0:
                    # Solving w(z)^2 = w0^2 * (1 + ((zt-z0)/zr)^2) for z0
                    z_dist = np.sqrt((w_t_target/w0)**2 - 1) * zr
                    z0 = zt - z_dist # Adjust focus pos
                    d['z0']['val'].value = round(z0, 2)
            # Case 2: Adjust Waist (fwhm0) if z0 is fixed
            else:
                new_w0 = get_beam_waist_from_fwhm(fwhm_t * 1e-3, zt, z0, wl_mm)
                if new_w0:
                    w0 = new_w0
                    d['fwhm0']['val'].value = round((w0 * self.w_to_fwhm) * 1e3, 4)

        return w0, z0, wl_mm

    def update(self, _=None):
        self.ax.clear()
        z_min, z_max = -4500, 4500
        z_range = np.linspace(z_min, z_max, int((z_max - z_min) * self.points_per_mm))
        
        for dim, color, lbl in zip(['x', 'y'], ['#1f77b4', '#d62728'], ['Hor (X)', 'Ver (Y)']):
            w0, z0, wl = self.calculate_beam(dim)
            zr = (np.pi * w0**2) / wl
            w_z_func = lambda z: w0 * np.sqrt(1 + ((z - z0) / zr)**2)
            
            fwhm_vals = w_z_func(z_range) * self.w_to_fwhm * 1e3
            self.ax.plot(z_range, fwhm_vals, color=color, lw=2, label=f'{lbl} FWHM')
            self.ax.plot(z_range, w_z_func(z_range) * 1e3, color=color, ls='--', alpha=0.3, label=f'{lbl} $1/e^2$')
            
            for row in self.z_rows:
                fwhm_size = w_z_func(row['z'].value) * self.w_to_fwhm * 1e3
                if dim == 'x': row['x'].value = round(fwhm_size, 2)
                else: row['y'].value = round(fwhm_size, 2)

        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            self.ax.axvline(row['z'].value, color=colors(i), lw=1.5, alpha=0.7, label=row['name'].value)

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Beam Size [µm]")
        self.ax.grid(True, which="both", alpha=0.2)
        self.ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
        self.canvas.draw_idle()

app = GaussianApp()
display(app.ui)

/tmp/ipykernel_511825/4183233962.py:162: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))


In [14]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

def energy_to_wl_mm(energy_kev):
    return 1.23984193e-6 / energy_kev

def solve_for_w0(fwhm_z, z, z0, wl_mm):
    """
    Solves the quadratic for w0^2 to find the waist radius.
    Returns the larger root (the stable propagation branch).
    """
    # Convert target FWHM width to 1/e2 radius w_z
    w_z = (fwhm_z * 1e-3) / (2 * np.sqrt(2 * np.log(2)))
    
    # Quadratic coefficients for (w0^2)^2 + B(w0^2) + C = 0
    # B = -w_z^2,  C = (lambda * delta_z / pi)^2
    C = (wl_mm * (z - z0) / np.pi)**2
    B = -(w_z**2)
    
    discriminant = B**2 - 4*C
    if discriminant < 0:
        return None # Physical impossibility: size at z is smaller than diffraction limit
    
    # Use the larger root for the waist
    w0_sq = (-B + np.sqrt(discriminant)) / 2
    return np.sqrt(w0_sq)

class GaussianApp:
    def __init__(self):
        self.w_to_fwhm = 2 * np.sqrt(2 * np.log(2))
        self.points_per_mm = 10
        self.is_solving = False 

        # Parameters: Note that fwhm0 is now the output of the solver
        self.inputs = {}
        for dim in ['x', 'y']:
            z_t_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'fwhm0': widgets.FloatText(value=0.0, description='Waist FWHM [µm]', disabled=True),
                'z0': widgets.FloatText(value=0.0, description='Focus $z_0$ [mm]'),
                'energy': widgets.FloatText(value=12.4, description='Energy [keV]'),
                'z_test': widgets.FloatText(value=z_t_def, description='Test $z$ [mm]'),
                'fwhm_test': widgets.FloatText(value=500.0, description='FWHM @ $z$ [µm]')
            }
            # All inputs (except the disabled waist) trigger the solver
            for key in ['z0', 'energy', 'z_test', 'fwhm_test']:
                self.inputs[dim][key].observe(lambda c, d=dim: self.run_solver(d), 'value')

        self.z_rows = []
        self.z_rows_container = widgets.VBox([])
        self.log_check = widgets.Checkbox(value=False, description='Log Scale')
        self.log_check.observe(lambda _: self.update_plot(), 'value')
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
        self.canvas = self.fig.canvas
        
        col_x = widgets.VBox([widgets.HTML("<b>Horizontal (X)</b>")] + list(self.inputs['x'].values()))
        col_y = widgets.VBox([widgets.HTML("<b>Vertical (Y)</b>")] + list(self.inputs['y'].values()))
        self.ui = widgets.VBox([widgets.HBox([col_x, col_y]), self.z_rows_container, self.log_check, self.canvas])
        
        self.load_beamline()
        # Initial run
        self.run_solver('x')
        self.run_solver('y')

    def run_solver(self, dim):
        if self.is_solving: return
        self.is_solving = True
        
        d = self.inputs[dim]
        wl = energy_to_wl_mm(d['energy'].value)
        
        w0_new = solve_for_w0(d['fwhm_test'].value, d['z_test'].value, d['z0'].value, wl)
        
        if w0_new is not None:
            d['fwhm0'].value = round(w0_new * self.w_to_fwhm * 1e3, 4)
            d['fwhm0'].background_color = 'white'
        else:
            # Visual feedback for invalid/impossible geometry
            d['fwhm0'].value = 0.0
            
        self.is_solving = False
        self.update_plot()

    def update_plot(self):
        self.ax.clear()
        z_pts = np.linspace(-4500, 4500, 2000)
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            d = self.inputs[dim]
            if d['fwhm0'].value <= 0: continue
            
            wl = energy_to_wl_mm(d['energy'].value)
            w0 = (d['fwhm0'].value * 1e-3) / self.w_to_fwhm
            zr = (np.pi * w0**2) / wl
            z0 = d['z0'].value
            
            w_z = lambda z: w0 * np.sqrt(1 + ((z - z0) / zr)**2)
            fwhm_z = w_z(z_pts) * self.w_to_fwhm * 1e3
            
            self.ax.plot(z_pts, fwhm_z, color=color, lw=2, label=f"{dim.upper()} FWHM")
            
            # Update Dynamic Probes
            for r in self.z_rows:
                val = round(w_z(r['z'].value) * self.w_to_fwhm * 1e3, 2)
                if dim == 'x': r['x'].value = val
                else: r['y'].value = val

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_ylabel("FWHM [µm]")
        self.ax.set_xlabel("z [mm]")
        self.ax.legend()
        self.ax.grid(True, which="both", alpha=0.2)
        self.canvas.draw_idle()

    def add_z_row(self, z, name):
        z_in = widgets.FloatText(value=z, layout={'width': '80px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '80px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '80px'})
        z_in.observe(lambda _: self.update_plot(), 'value')
        row = widgets.HBox([widgets.Label(name, layout={'width': '150px'}), z_in, x_out, y_out])
        self.z_rows.append({'z': z_in, 'x': x_out, 'y': y_out})
        self.z_rows_container.children = list(self.z_rows_container.children) + [row]

    def load_beamline(self):
        defaults = [("KB Mirror", -3000), ("Aperture", -1500), ("Sample", -9)]
        for n, z in defaults:
            self.add_z_row(z, n)

app = GaussianApp()
display(app.ui)

In [15]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

class GeometricBeamApp:
    def __init__(self):
        # UI Elements for X and Y
        self.inputs = {}
        for dim in ['x', 'y']:
            z_t_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'z0': widgets.FloatText(value=0.0, description=f'Focus $z_0$ [mm]'),
                'z_test': widgets.FloatText(value=z_t_def, description=f'Test $z$ [mm]'),
                'w_test': widgets.FloatText(value=500.0, description=f'FWHM @ $z$ [µm]')
            }
            # Attach observers
            for w in self.inputs[dim].values():
                w.observe(lambda _: self.update_plot(), 'value')

        # Dynamic Rows for Z-Positions (Components)
        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        
        self.log_check = widgets.Checkbox(value=False, description='Log Scale', layout={'width': '150px'})
        self.log_check.observe(lambda _: self.update_plot(), 'value')

        # Layout Building
        col_x = widgets.VBox([widgets.HTML("<b>Horizontal (X)</b>")] + list(self.inputs['x'].values()))
        col_y = widgets.VBox([widgets.HTML("<b>Vertical (Y)</b>")] + list(self.inputs['y'].values()))
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(11, 5))
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            widgets.HTML("<h2>Geometric Beam Scaling Applet</h2>"),
            widgets.HBox([col_x, col_y]), 
            widgets.HTML("<b>Beamline Components (Z-Probes)</b>"),
            self.z_rows_container,
            self.log_check,
            self.canvas
        ])
        
        self.load_defaults()
        self.update_plot()

    def add_z_row(self, initial_z=0.0, initial_name="New Point"):
        name_lbl = widgets.Label(value=initial_name, layout={'width': '150px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        
        z_in.observe(lambda _: self.update_plot(), 'value')
        
        row_ui = widgets.HBox([name_lbl, widgets.Label("Z:"), z_in, 
                               widgets.Label("X size:"), x_out, widgets.Label("Y size:"), y_out])
        
        self.z_rows.append({'row': row_ui, 'z': z_in, 'x': x_out, 'y': y_out, 'name': initial_name})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]

    def update_plot(self, _=None):
        self.ax.clear()
        z_range = np.linspace(-4500, 4500, 2000)
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            d = self.inputs[dim]
            z0 = d['z0'].value
            zt = d['z_test'].value
            wt = d['w_test'].value
            
            # Geometric scaling formula: size(z) = size_test * |(z - z0) / (zt - z0)|
            # Prevent division by zero if focus and test position coincide
            denom = (zt - z0) if (zt - z0) != 0 else 1e-9
            w_z_func = lambda z: wt * np.abs((z - z0) / denom)
            
            sizes = w_z_func(z_range)
            self.ax.plot(z_range, sizes, color=color, lw=2, label=f'{dim.upper()} Beam')
            
            # Update values in the UI rows
            for row in self.z_rows:
                val = round(w_z_func(row['z'].value), 2)
                if dim == 'x': row['x'].value = val
                else: row['y'].value = val

        # Draw vertical lines for components
        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            self.ax.axvline(row['z'].value, color=colors(i), lw=1, alpha=0.5)

        self.ax.set_yscale('log' if self.log_check.value else 'linear')
        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Beam Size [µm]")
        self.ax.grid(True, which="both", alpha=0.2)
        self.ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
        self.canvas.draw_idle()

    def load_defaults(self):
        defaults = [
            ("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
            ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
            ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
            ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)
        ]
        for name, z_pos in defaults:
            self.add_z_row(z_pos, name)

app = GeometricBeamApp()
display(app.ui)

/tmp/ipykernel_511825/1072763554.py:89: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))


In [17]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

class GeometricBeamApp:
    def __init__(self):
        # UI Elements for X and Y
        self.inputs = {}
        for dim in ['x', 'y']:
            z_t_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'z0': widgets.FloatText(value=0.0, description=f'Focus $z_0$ [mm]'),
                'z_test': widgets.FloatText(value=z_t_def, description=f'Test $z$ [mm]'),
                'w_test': widgets.FloatText(value=500.0, description=f'FWHM @ $z$ [µm]')
            }
            # Attach observers for automatic plot updates
            for w in self.inputs[dim].values():
                w.observe(lambda _: self.update_plot(), 'value')

        # Dynamic Rows for Z-Positions (Components)
        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        
        # Management Buttons
        self.add_row_btn = widgets.Button(description="+ Add Component", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove Last", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        # Layout Building
        col_x = widgets.VBox([widgets.HTML("<b>Horizontal (X)</b>")] + list(self.inputs['x'].values()))
        col_y = widgets.VBox([widgets.HTML("<b>Vertical (Y)</b>")] + list(self.inputs['y'].values()))
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            widgets.HTML("<h2>Geometric Beamline Propagation Tool</h2>"),
            widgets.HBox([col_x, col_y]), 
            widgets.HTML("<b>Beamline Components (Z-Probes)</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn]),
            self.canvas
        ])
        
        self.load_defaults()
        self.update_plot()

    def add_z_row(self, initial_z=0.0, initial_name="New Component"):
        name_in = widgets.Text(value=initial_name, layout={'width': '150px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        y_out = widgets.FloatText(value=0, disabled=True, layout={'width': '90px'})
        
        name_in.observe(lambda _: self.update_plot(), 'value')
        z_in.observe(lambda _: self.update_plot(), 'value')
        
        row_ui = widgets.HBox([widgets.Label("Name:"), name_in, widgets.Label("Z:"), z_in, 
                               widgets.Label("X size:"), x_out, widgets.Label("Y size:"), y_out])
        
        self.z_rows.append({'row': row_ui, 'name_widget': name_in, 'z': z_in, 'x': x_out, 'y': y_out})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update_plot()

    def remove_z_row(self, _=None):
        if len(self.z_rows) > 0:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update_plot()

    def export_to_csv(self, _=None):
        data = []
        for r in self.z_rows:
            data.append([r['name_widget'].value, r['z'].value, r['x'].value, r['y'].value])
        df = pd.DataFrame(data, columns=['Component', 'Z_Pos_mm', 'X_FWHM_um', 'Y_FWHM_um'])
        df.to_csv("beamline_geometric_export.csv", index=False)
        print("Exported current probe data to beamline_geometric_export.csv")

    def update_plot(self, _=None):
        self.ax.clear()
        z_range = np.linspace(-4500, 4500, 2000)
        all_max_sizes = [100] # Seed with default to avoid empty errors
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            d = self.inputs[dim]
            z0 = d['z0'].value
            zt = d['z_test'].value
            wt = d['w_test'].value
            
            denom = (zt - z0) if (zt - z0) != 0 else 1e-9
            w_z_func = lambda z: wt * np.abs((z - z0) / denom)
            
            sizes = w_z_func(z_range)
            all_max_sizes.append(np.max(sizes))
            self.ax.plot(z_range, sizes, color=color, lw=2, label=f'{dim.upper()} Beam')
            
            for row in self.z_rows:
                val = round(w_z_func(row['z'].value), 2)
                if dim == 'x': row['x'].value = val
                else: row['y'].value = val

        y_limit = max(all_max_sizes)
        
        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            z_pos = row['z'].value
            comp_name = row['name_widget'].value
            self.ax.axvline(z_pos, color=colors(i), lw=1.5, alpha=0.5, label=comp_name)
            self.ax.text(z_pos, y_limit * 0.98, comp_name, rotation=90, 
                         verticalalignment='top', horizontalalignment='right',
                         fontsize=8, color=colors(i), fontweight='bold')

        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Beam Size [µm]")
        self.ax.set_title("Geometric Beamline Propagation")
        self.ax.grid(True, which="major", alpha=0.3)
        self.ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
        self.canvas.draw_idle()

    def load_defaults(self):
        defaults = [
            ("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
            ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
            ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
            ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)
        ]
        for name, z_pos in defaults:
            self.add_z_row(z_pos, name)

app = GeometricBeamApp()
display(app.ui)

/tmp/ipykernel_511825/2770415329.py:111: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))


In [19]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

class GeometricBeamApp:
    def __init__(self):
        self.is_updating = False 
        
        # UI Elements for X and Y
        self.inputs = {}
        for dim in ['x', 'y']:
            z_t_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'z0': widgets.FloatText(value=0.0, description=f'Focus $z_0$ [mm]'),
                'z_test': widgets.FloatText(value=z_t_def, description=f'Test $z$ [mm]'),
                'w_test': widgets.FloatText(value=500.0, description=f'Test FWHM [µm]')
            }
            for w in self.inputs[dim].values():
                w.observe(self.on_global_input_change, 'value')

        self.z_rows_container = widgets.VBox([])
        self.z_rows = []
        
        self.add_row_btn = widgets.Button(description="+ Add Component", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove Last", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        col_x = widgets.VBox([widgets.HTML("<b>Horizontal (X)</b>")] + list(self.inputs['x'].values()))
        col_y = widgets.VBox([widgets.HTML("<b>Vertical (Y)</b>")] + list(self.inputs['y'].values()))
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
        self.canvas = self.fig.canvas

        self.ui = widgets.VBox([
            widgets.HTML("<h2>Geometric Signed Propagation Tool</h2>"),
            widgets.HBox([col_x, col_y]), 
            widgets.HTML("<b>Beamline Components (Z-Probes)</b>"),
            widgets.HTML("<small>Signed Size: Negative = Before Focus | Positive = After Focus</small>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn]),
            self.canvas
        ])
        
        self.load_defaults()
        self.update_plot()

    def on_global_input_change(self, _):
        if self.is_updating: return
        self.update_plot()

    def on_component_size_change(self, change, dim, row_idx):
        if self.is_updating: return
        self.is_updating = True
        
        try:
            row = self.z_rows[row_idx]
            d = self.inputs[dim]
            
            z_comp = row['z'].value
            w_comp_signed = change['new']
            z_test = d['z_test'].value
            w_test = d['w_test'].value
            
            # The signed logic: w_signed = w_abs * sign(z - z0)
            # In geometric scaling: w_comp / w_test = (z_comp - z0) / (z_test - z0)
            # Solving for z0:
            ratio = w_comp_signed / w_test if w_test != 0 else 0
            
            if ratio != 1:
                # Linear solution: z0 = (z_comp - ratio * z_test) / (1 - ratio)
                new_z0 = (z_comp - ratio * z_test) / (1 - ratio)
                d['z0'].value = round(new_z0, 2)
        finally:
            self.is_updating = False
            self.update_plot()

    def add_z_row(self, initial_z=0.0, initial_name="New Component"):
        idx = len(self.z_rows)
        name_in = widgets.Text(value=initial_name, layout={'width': '150px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '100px'})
        x_in = widgets.FloatText(value=0, layout={'width': '90px'})
        y_in = widgets.FloatText(value=0, layout={'width': '90px'})
        
        name_in.observe(lambda _: self.update_plot(), 'value')
        z_in.observe(lambda _: self.update_plot(), 'value')
        
        x_in.observe(lambda c: self.on_component_size_change(c, 'x', idx), 'value')
        y_in.observe(lambda c: self.on_component_size_change(c, 'y', idx), 'value')
        
        row_ui = widgets.HBox([widgets.Label("Name:"), name_in, widgets.Label("Z:"), z_in, 
                               widgets.Label("X size:"), x_in, widgets.Label("Y size:"), y_in])
        
        self.z_rows.append({'row': row_ui, 'name_widget': name_in, 'z': z_in, 'x': x_in, 'y': y_in})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update_plot()

    def remove_z_row(self, _=None):
        if len(self.z_rows) > 0:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update_plot()

    def export_to_csv(self, _=None):
        data = [[r['name_widget'].value, r['z'].value, r['x'].value, r['y'].value] for r in self.z_rows]
        df = pd.DataFrame(data, columns=['Component', 'Z_Pos_mm', 'X_Signed_Size_um', 'Y_Signed_Size_um'])
        df.to_csv("beamline_signed_export.csv", index=False)

    def update_plot(self, _=None):
        self.ax.clear()
        z_range = np.linspace(-4500, 4500, 2000)
        all_max_sizes = [100]
        
        # Semaphore to prevent update loops
        old_updating = self.is_updating
        self.is_updating = True
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            d = self.inputs[dim]
            z0, zt, wt = d['z0'].value, d['z_test'].value, d['w_test'].value
            
            # The test size 'wt' is always assumed to be on the 'positive' (post-focus) 
            # side for calculation logic, OR we deduce its sign from zt - z0
            test_sign = np.sign(zt - z0) if zt != z0 else 1
            
            # Linear scaling function (preserving sign)
            # w(z) = w_test * (z - z0) / (zt - z0)
            denom = (zt - z0) if (zt - z0) != 0 else 1e-9
            w_z_signed_func = lambda z: wt * (z - z0) / denom
            
            # Plot absolute values
            sizes_abs = np.abs(w_z_signed_func(z_range))
            all_max_sizes.append(np.max(sizes_abs))
            self.ax.plot(z_range, sizes_abs, color=color, lw=2, label=f'{dim.upper()} Beam (Absolute)')
            
            # Update component rows with signed values
            for row in self.z_rows:
                val = w_z_signed_func(row['z'].value)
                row[dim].value = round(val, 2)

        self.is_updating = old_updating
        
        y_limit = max(all_max_sizes)
        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            z_pos, name = row['z'].value, row['name_widget'].value
            self.ax.axvline(z_pos, color=colors(i), lw=1.5, alpha=0.5, label=name)
            self.ax.text(z_pos, y_limit * 0.98, name, rotation=90, va='top', ha='right',
                         fontsize=8, color=colors(i), fontweight='bold')

        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Absolute Beam Size [µm]")
        self.ax.grid(True, which="major", alpha=0.3)
        self.ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
        self.canvas.draw_idle()

    def load_defaults(self):
        defaults = [("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
                    ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
                    ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
                    ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)]
        for name, z_pos in defaults: self.add_z_row(z_pos, name)

app = GeometricBeamApp()
display(app.ui)

/tmp/ipykernel_511825/1497925064.py:151: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))


In [20]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

def energy_to_wl_mm(energy_kev):
    return 1.23984193e-6 / energy_kev

class GeometricBeamApp:
    def __init__(self):
        self.is_updating = False 
        self.w_to_fwhm = 2 * np.sqrt(2 * np.log(2))
        self.fwhm_to_w = 1 / self.w_to_fwhm
        
        # Core Parameters
        self.inputs = {}
        self.derived = {}
        
        for dim in ['x', 'y']:
            z_t_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'z0': widgets.FloatText(value=0.0, description=f'Focus $z_0$ [mm]'),
                'z_test': widgets.FloatText(value=z_t_def, description=f'Test $z$ [mm]'),
                'w_test': widgets.FloatText(value=500.0, description=f'Test FWHM [µm]'),
                'energy': widgets.FloatText(value=12.4, description=f'Energy [keV]')
            }
            self.derived[dim] = {
                'div': widgets.FloatText(value=0.0, description=f'Div (FWHM) [mrad]', disabled=True),
                'w0': widgets.FloatText(value=0.0, description=f'Waist $w_0$ [µm]', disabled=True),
                'zr': widgets.FloatText(value=0.0, description=f'Rayleigh $z_R$ [mm]', disabled=True)
            }
            for w in self.inputs[dim].values():
                w.observe(self.on_global_input_change, 'value')

        self.z_rows_container = widgets.VBox([], layout={'max_height': '400px', 'overflow_y': 'auto'})
        self.z_rows = []
        self.setup_ui()
        self.load_defaults()
        self.update_plot()

    def on_global_input_change(self, _):
        if self.is_updating: return
        self.update_plot()

    def solve_optics(self, dim):
        d = self.inputs[dim]
        # 1. Geometric Divergence
        dist = abs(d['z_test'].value - d['z0'].value)
        if dist == 0: dist = 1e-9
        # Full angle FWHM: 2 * arctan(half_width / dist)
        div_fwhm_rad = 2 * np.arctan((d['w_test'].value * 1e-3 / 2) / dist)
        self.derived[dim]['div'].value = round(div_fwhm_rad * 1000, 4) # mrad
        
        # 2. Gaussian Parameters
        wl = energy_to_wl_mm(d['energy'].value)
        # Convert FWHM divergence to 1/e2 divergence radius (half-angle)
        theta_1e2_rad = (div_fwhm_rad / 2) / (self.w_to_fwhm / 2)
        
        # w0 = lambda / (pi * theta)
        w0_mm = wl / (np.pi * theta_1e2_rad)
        zr_mm = (np.pi * w0_mm**2) / wl
        
        self.derived[dim]['w0'].value = round(w0_mm * 1e3, 4)
        self.derived[dim]['zr'].value = round(zr_mm, 4)
        return w0_mm, wl

    def setup_ui(self):
        self.add_row_btn = widgets.Button(description="+ Add Component", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove Last", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        # Build Parameter Panels
        panels = []
        for dim in ['x', 'y']:
            title = "Horizontal (X)" if dim == 'x' else "Vertical (Y)"
            inp_list = list(self.inputs[dim].values())
            der_list = list(self.derived[dim].values())
            panels.append(widgets.VBox([
                widgets.HTML(f"<b>{title}</b>"),
                widgets.VBox(inp_list),
                widgets.HTML("<i>Calculated Optics</i>"),
                widgets.VBox(der_list)
            ], layout={'border': '1px solid #ccc', 'padding': '10px', 'margin': '5px', 'width': '300px'}))
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(8, 5))
        self.canvas = self.fig.canvas

        # Floating Layout Logic
        # Left side: Parameters and Component List
        left_box = widgets.VBox([
            widgets.HBox(panels),
            widgets.HTML("<b>Beamline Components</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn])
        ], layout={'min_width': '650px'})
        
        # Combine Left (Params) and Right (Figure)
        self.ui = widgets.HBox([left_box, self.canvas], layout={'width': '100%', 'flex_flow': 'row wrap'})

    def on_component_size_change(self, change, dim, row_idx):
        if self.is_updating: return
        self.is_updating = True
        try:
            row = self.z_rows[row_idx]
            d = self.inputs[dim]
            ratio = change['new'] / d['w_test'].value if d['w_test'].value != 0 else 0
            if ratio != 1:
                new_z0 = (row['z'].value - ratio * d['z_test'].value) / (1 - ratio)
                d['z0'].value = round(new_z0, 2)
        finally:
            self.is_updating = False
            self.update_plot()

    def add_z_row(self, initial_z=0.0, initial_name="New Component"):
        idx = len(self.z_rows)
        name_in = widgets.Text(value=initial_name, layout={'width': '140px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '80px'})
        x_in = widgets.FloatText(value=0, layout={'width': '80px'})
        y_in = widgets.FloatText(value=0, layout={'width': '80px'})
        
        z_in.observe(lambda _: self.update_plot(), 'value')
        x_in.observe(lambda c: self.on_component_size_change(c, 'x', idx), 'value')
        y_in.observe(lambda c: self.on_component_size_change(c, 'y', idx), 'value')
        
        row_ui = widgets.HBox([name_in, z_in, x_in, y_in])
        self.z_rows.append({'row': row_ui, 'name': name_in, 'z': z_in, 'x': x_in, 'y': y_in})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update_plot()

    def remove_z_row(self, _=None):
        if self.z_rows:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update_plot()

    def export_to_csv(self, _=None):
        data = [[r['name'].value, r['z'].value, r['x'].value, r['y'].value] for r in self.z_rows]
        pd.DataFrame(data, columns=['Comp', 'Z_mm', 'X_um', 'Y_um']).to_csv("beamline.csv", index=False)

    def update_plot(self, _=None):
        self.ax.clear()
        z_range = np.linspace(-4500, 4500, 2000)
        self.is_updating = True
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            self.solve_optics(dim)
            d = self.inputs[dim]
            denom = (d['z_test'].value - d['z0'].value) or 1e-9
            w_z_func = lambda z: d['w_test'].value * (z - d['z0'].value) / denom
            
            self.ax.plot(z_range, np.abs(w_z_func(z_range)), color=color, lw=2, label=f'{dim.upper()} Beam')
            for row in self.z_rows:
                row[dim].value = round(w_z_func(row['z'].value), 2)

        self.is_updating = False
        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Size [µm]")
        self.ax.legend(loc='upper right')
        self.ax.grid(True, alpha=0.3)
        self.canvas.draw_idle()

    def load_defaults(self):
        defaults = [("KB ver", -3350.0), ("KB hor", -2600.0), ("Sample", -9.0), ("Detector", 3725.0)]
        for n, z in defaults: self.add_z_row(z, n)

app = GeometricBeamApp()
display(app.ui)

In [3]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

def energy_to_wl_mm(energy_kev):
    return 1.23984193e-6 / energy_kev

class GeometricBeamApp:
    def __init__(self):
        self.is_updating = False 
        self.w_to_fwhm = 2 * np.sqrt(2 * np.log(2))
        self.fwhm_to_w = 1 / self.w_to_fwhm
        
        # UI Styling
        self.lbl_layout = widgets.Layout(width='66%')
        self.inp_layout = widgets.Layout(width='33%')
        
        # Core Parameters
        self.inputs = {}
        self.derived = {}
        
        for dim in ['x', 'y']:
            z_t_def = -2600.0 if dim == 'x' else -3350.0
            self.inputs[dim] = {
                'z0': widgets.FloatText(value=0.0),
                'z_test': widgets.FloatText(value=z_t_def),
                'w_test': widgets.FloatText(value=500.0),
                'energy': widgets.FloatText(value=12.4)
            }
            self.derived[dim] = {
                'div': widgets.FloatText(value=0.0, disabled=True),
                'w0': widgets.FloatText(value=0.0, disabled=True),
                'zr': widgets.FloatText(value=0.0, disabled=True)
            }
            for w in self.inputs[dim].values():
                w.observe(self.on_global_input_change, 'value')

        self.z_rows_container = widgets.VBox([], layout={'max_height': '400px', 'overflow_y': 'auto'})
        self.z_rows = []
        
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(8, 5),constrained_layout=True)
        self.canvas = self.fig.canvas
        
        self.setup_ui()
        self.load_defaults()
        self.update_plot()

    def on_global_input_change(self, _):
        if self.is_updating: return
        self.update_plot()

    def solve_optics(self, dim):
        d = self.inputs[dim]
        dist = abs(d['z_test'].value - d['z0'].value) or 1e-9
        div_fwhm_rad = 2 * np.arctan((d['w_test'].value * 1e-3 / 2) / dist)
        self.derived[dim]['div'].value = round(div_fwhm_rad * 1000, 4)
        
        wl = energy_to_wl_mm(d['energy'].value)
        theta_1e2_rad = (div_fwhm_rad / 2) / (self.w_to_fwhm / 2)
        w0_mm = wl / (np.pi * theta_1e2_rad) if theta_1e2_rad != 0 else 0
        zr_mm = (np.pi * w0_mm**2) / wl if wl != 0 else 0
        
        self.derived[dim]['w0'].value = round(w0_mm * 1e3, 4)
        self.derived[dim]['zr'].value = round(zr_mm, 4)
        return w0_mm, wl

    def create_param_row(self, label, widget):
        return widgets.HBox([widgets.Label(label, layout=self.lbl_layout), widget], layout={'width': '100%'})

    def setup_ui(self):
        self.add_row_btn = widgets.Button(description="+ Add Component", button_style='success')
        self.rem_row_btn = widgets.Button(description="- Remove Last", button_style='danger')
        self.export_btn = widgets.Button(description="📥 Export CSV", button_style='info')
        
        self.add_row_btn.on_click(lambda _: self.add_z_row())
        self.rem_row_btn.on_click(lambda _: self.remove_z_row())
        self.export_btn.on_click(self.export_to_csv)
        
        panels = []
        labels_in = ["Focus $z_0$ [mm]", "Test $z$ [mm]", "Test FWHM [µm]", "Energy [keV]"]
        labels_der = ["Div (FWHM) [mrad]", "Waist $w_0$ [µm]", "Rayleigh $z_R$ [mm]"]
        
        for dim in ['x', 'y']:
            title = "Horizontal (X)" if dim == 'x' else "Vertical (Y)"
            rows = [widgets.HTML(f"<b>{title}</b>")]
            for lbl, key in zip(labels_in, self.inputs[dim].keys()):
                rows.append(self.create_param_row(lbl, self.inputs[dim][key]))
            rows.append(widgets.HTML("<i>Calculated Optics</i>"))
            for lbl, key in zip(labels_der, self.derived[dim].keys()):
                rows.append(self.create_param_row(lbl, self.derived[dim][key]))
            panels.append(widgets.VBox(rows, layout={'border': '1px solid #ccc', 'padding': '10px', 'margin': '5px', 'width': '320px'}))
        
        left_box = widgets.VBox([
            widgets.HBox(panels),
            widgets.HTML("<b>Beamline Components</b>"),
            self.z_rows_container,
            widgets.HBox([self.add_row_btn, self.rem_row_btn, self.export_btn])
        ], layout={'min_width': '660px'})
        
        self.ui = widgets.HBox([left_box, self.canvas], layout={'width': '100%', 'flex_flow': 'row wrap'})

    def on_component_size_change(self, change, dim, row_idx):
        if self.is_updating: return
        self.is_updating = True
        try:
            row = self.z_rows[row_idx]
            d = self.inputs[dim]
            ratio = change['new'] / d['w_test'].value if d['w_test'].value != 0 else 0
            if ratio != 1:
                new_z0 = (row['z'].value - ratio * d['z_test'].value) / (1 - ratio)
                d['z0'].value = round(new_z0, 2)
        finally:
            self.is_updating = False
            self.update_plot()

    def add_z_row(self, initial_z=0.0, initial_name="New"):
        idx = len(self.z_rows)
        name_in = widgets.Text(value=initial_name, layout={'width': '140px'})
        z_in = widgets.FloatText(value=initial_z, layout={'width': '80px'})
        x_in = widgets.FloatText(value=0, layout={'width': '80px'})
        y_in = widgets.FloatText(value=0, layout={'width': '80px'})
        
        z_in.observe(lambda _: self.update_plot(), 'value')
        x_in.observe(lambda c: self.on_component_size_change(c, 'x', idx), 'value')
        y_in.observe(lambda c: self.on_component_size_change(c, 'y', idx), 'value')
        
        row_ui = widgets.HBox([name_in, z_in, x_in, y_in])
        self.z_rows.append({'row': row_ui, 'name': name_in, 'z': z_in, 'x': x_in, 'y': y_in})
        self.z_rows_container.children = [r['row'] for r in self.z_rows]
        self.update_plot()

    def remove_z_row(self, _=None):
        if self.z_rows:
            self.z_rows.pop()
            self.z_rows_container.children = [r['row'] for r in self.z_rows]
            self.update_plot()

    def export_to_csv(self, _=None):
        data = [[r['name'].value, r['z'].value, r['x'].value, r['y'].value] for r in self.z_rows]
        pd.DataFrame(data, columns=['Comp', 'Z_mm', 'X_um', 'Y_um']).to_csv("beamline.csv", index=False)

    def update_plot(self, _=None):
        self.ax.clear()
        z_range = np.linspace(-4500, 4500, 2000)
        self.is_updating = True
        
        for dim, color in zip(['x', 'y'], ['#1f77b4', '#d62728']):
            self.solve_optics(dim)
            d = self.inputs[dim]
            denom = (d['z_test'].value - d['z0'].value) or 1e-9
            w_z_func = lambda z: d['w_test'].value * (z - d['z0'].value) / denom
            
            self.ax.plot(z_range, np.abs(w_z_func(z_range)), color=color, lw=2, label=f'{dim.upper()} Beam')
            for row in self.z_rows:
                row[dim].value = round(w_z_func(row['z'].value), 2)

        self.is_updating = False
        # Component Lines
        colors = plt.cm.get_cmap('tab20', len(self.z_rows))
        for i, row in enumerate(self.z_rows):
            self.ax.axvline(row['z'].value, color=colors(i), alpha=0.5, label=row['name'].value)

        self.ax.set_xlabel("z [mm]")
        self.ax.set_ylabel("Size [µm]")
        self.ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
        self.ax.grid(True, alpha=0.3)
        self.canvas.draw_idle()

    def load_defaults(self):
        defaults = [
            ("KB ver", -3350.0), ("KB hor", -2600.0), ("C* win 1", -1945.0),
            ("slit_kb", -1850.0), ("mon_kb target", -1750.0), ("prof_kb/tt_kb target", -1750.0),
            ("att_usd", -1420.0), ("C* win 2", -1330.0), ("sample", -9.0),
            ("plate_mondsd", 3147.5), ("prof_dsd", 3725.0)
        ]
        for n, z in defaults: self.add_z_row(z, n)

app = GeometricBeamApp()
display(app.ui)

/tmp/ipykernel_92435/93807792.py:163: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20', len(self.z_rows))
